# Notebook 6W — Context + Embedding RoBERTa (Women Discourse)

This notebook trains the first context-augmented model.

It uses the reusable mapping produced by the Wikimedia Context Bridge notebook:

```text
women_context_artifacts/context_mapped_examples.parquet
```

Model design:

```text
Retrieved Wikimedia context + comment
        ↓
      RoBERTa
        ↓
   CLS representation

subgroup_id
        ↓
subgroup embedding

[CLS ; subgroup_embedding]
        ↓
 classifier
        ↓
3-class probability distribution
```

This is the direct context version of the previous subgroup-embedding baseline.

The key question:

> Does retrieved women-discourse Wikimedia context improve distributional hate-speech prediction compared with the no-context embedding baseline?


## 1. Imports and Configuration

In [19]:
import ast
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42

MODEL_NAME = "roberta-base"
NUM_LABELS = 3

# Context inputs are longer. RTX 4060 8GB should usually handle this.
MAX_LENGTH = 384
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

SUBGROUP_DIM = 32
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Women-discourse subgroup restriction.
# This excludes prefer_not_to_say and self_describe from the model pipeline.
ALLOWED_SUBGROUPS = ["women", "men", "non_binary"]

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebook/data")
CONTEXT_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet")

OUTPUT_DIR = Path("context_embedding_women_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Context file:", CONTEXT_PATH)
print("Output directory:", OUTPUT_DIR.resolve())


Device: cuda
Context file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/context_embedding_women_outputs


In [20]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Context-Mapped Dataset

In [21]:
context_df = pd.read_parquet(CONTEXT_PATH)

# Keep only the three gender subgroups used in the women-discourse experiments.
before_filter = len(context_df)
context_df = (
    context_df[context_df["subgroup"].isin(ALLOWED_SUBGROUPS)]
    .copy()
    .reset_index(drop=True)
)
print(f"Filtered to allowed subgroups {ALLOWED_SUBGROUPS}: {before_filter:,} -> {len(context_df):,} rows")

print("Context dataset:", context_df.shape)
display(context_df.head(2))

required_columns = [
    "comment_id",
    "split",
    "subgroup",
    "text",
    "target_distribution",
    "target_majority_label",
    "context_input_text",
    "retrieved_article_titles",
    "retrieved_summaries",
]

missing = [col for col in required_columns if col not in context_df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Columns OK.")


Filtered to allowed subgroups ['women', 'men', 'non_binary']: 7,962 -> 7,962 rows
Context dataset: (7962, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.10770449787378311],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and prac...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BAC...,35,195
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[0.07197389751672745],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from ...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED B...,35,189


Columns OK.


## 3. Sanity Checks on Context Mapping

In [22]:
print("Rows by split:")
display(context_df["split"].value_counts())

print("\nRows by subgroup:")
display(context_df["subgroup"].value_counts())

print("\nTarget majority distribution:")
display(context_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("\nMissing context text:", context_df["context_input_text"].isna().sum())
print("Missing target distributions:", context_df["target_distribution"].isna().sum())


Rows by split:


split
train         5565
validation    1224
test          1173
Name: count, dtype: int64


Rows by subgroup:


subgroup
women         3919
men           3903
non_binary     140
Name: count, dtype: int64


Target majority distribution:


target_majority_label
0    0.677594
1    0.073851
2    0.248556
Name: proportion, dtype: float64


Missing context text: 0
Missing target distributions: 0


In [23]:
def safe_len(x):
    if isinstance(x, list):
        return len(x)
    if isinstance(x, np.ndarray):
        return len(x)
    if isinstance(x, str):
        try:
            return len(ast.literal_eval(x))
        except Exception:
            return np.nan
    return np.nan

print("Retrieved article count per row:")
display(context_df["retrieved_article_titles"].apply(safe_len).value_counts(dropna=False))

print("Retrieved summary count per row:")
display(context_df["retrieved_summaries"].apply(safe_len).value_counts(dropna=False))

print("\nExample full context input:")
print("=" * 100)
print(context_df.iloc[0]["context_input_text"])


Retrieved article count per row:


retrieved_article_titles
1    7962
Name: count, dtype: int64

Retrieved summary count per row:


retrieved_summaries
1    7962
Name: count, dtype: int64


Example full context input:
### COMMENT TO CLASSIFY
First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Male privilege:
In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, leading them to attribute their status to individual merits rather than unearned advantages. The ideal masculine norm varies by society but is often associated with being white, heterosexual, stoic, wealthy, strong, tough, competitive, and autonomous. Men who deviate from this norm may not benefit fully from privilege, while those who conform to it are more likely to receive rewards and recognition.


## 4. Prepare Subgroup IDs and Splits

In [24]:
subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(context_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

context_df["subgroup_id"] = context_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

train_df = context_df[context_df["split"] == "train"].reset_index(drop=True)
val_df = context_df[context_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = context_df[context_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

display(pd.crosstab(context_df["split"], context_df["subgroup"]))


Subgroup mapping:
{'men': 0, 'non_binary': 1, 'women': 2}
Train: (5565, 17)
Val: (1224, 17)
Test: (1173, 17)


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


## 5. Token Length Checks

In [25]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )

if "context_input_token_length" not in context_df.columns:
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_tokens)

length_summary = {
    "mean": context_df["context_input_token_length"].mean(),
    "median": context_df["context_input_token_length"].median(),
    "p95": context_df["context_input_token_length"].quantile(0.95),
    "max": context_df["context_input_token_length"].max(),
    "pct_over_512": float((context_df["context_input_token_length"] > 512).mean()),
}

display(pd.DataFrame([length_summary]))

print("Rows over MAX_LENGTH:", (context_df["context_input_token_length"] > MAX_LENGTH).sum())


,mean,median,p95,max,pct_over_512
0,170.833459,170.0,223.0,296,0.0


Rows over MAX_LENGTH: 0


## 6. Dataset and Dataloader

In [26]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        return [float(x) for x in parsed]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class ContextEmbeddingDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["context_input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        target_distribution = parse_distribution(row["target_distribution"])

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(target_distribution, dtype=torch.float),
        }


train_dataset = ContextEmbeddingDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = ContextEmbeddingDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = ContextEmbeddingDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([8, 384]),
 'attention_mask': torch.Size([8, 384]),
 'subgroup_id': torch.Size([8]),
 'target_distribution': torch.Size([8, 3])}

## 7. Context + Embedding Model

In [27]:
class ContextEmbeddingRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.subgroup_embedding = nn.Embedding(
            num_embeddings=num_subgroups,
            embedding_dim=subgroup_dim,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size + subgroup_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        combined = torch.cat(
            [cls_embedding, subgroup_embedding],
            dim=-1,
        )

        logits = self.classifier(combined)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


model = ContextEmbeddingRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Train batches per epoch:", len(train_loader))
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Optimizer update steps per epoch:", num_update_steps_per_epoch)
print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train batches per epoch: 696
Gradient accumulation: 1
Optimizer update steps per epoch: 696
Training steps: 6960
Warmup steps: 696


## 8. Metrics

In [28]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 9. Training Helpers

In [29]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                subgroup_id=subgroup_id,
            )

            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                should_step = (
                    ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0)
                    or ((step + 1) == len(dataloader))
                )

                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()

                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 10. Train Model

In [30]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "context_embedding_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / "context_embedding_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/10
Train KL: 0.7138 | Val KL: 0.7006
Train JS: 0.2850 | Val JS: 0.2427
Val Acc: 0.7132 | Val Macro F1: 0.4652
Saved new best model to context_embedding_women_outputs/context_embedding_best_model.pt
Epoch 2/10
Train KL: 0.6251 | Val KL: 0.6521
Train JS: 0.2352 | Val JS: 0.2394
Val Acc: 0.7116 | Val Macro F1: 0.4759
Saved new best model to context_embedding_women_outputs/context_embedding_best_model.pt
Epoch 3/10
Train KL: 0.5803 | Val KL: 0.6684
Train JS: 0.2177 | Val JS: 0.2352
Val Acc: 0.7124 | Val Macro F1: 0.4654
Epoch 4/10
Train KL: 0.5437 | Val KL: 0.7320
Train JS: 0.2037 | Val JS: 0.2448
Val Acc: 0.7100 | Val Macro F1: 0.4377
Epoch 5/10
Train KL: 0.5021 | Val KL: 0.7181
Train JS: 0.1882 | Val JS: 0.2414
Val Acc: 0.6977 | Val Macro F1: 0.4682
Epoch 6/10
Train KL: 0.4626 | Val KL: 0.7524
Train JS: 0.1733 | Val JS: 0.2462
Val Acc: 0.7132 | Val Macro F1: 0.4626
Epoch 7/10
Train KL: 0.4239 | Val KL: 0.8395
Train JS: 0.1594 | Val JS: 0.2459
Val Acc: 0.7034 | Val Macro F1: 0.437

,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.713764,0.284970,0.794371,0.701887,0.407967,0.656423,0.087376,0.072332,0.713764,0.700550,0.242652,0.773680,0.713235,0.465157,0.532047,0.106230,0.117144,0.700550
1,2,0.625088,0.235211,0.705694,0.723270,0.467153,0.535619,0.124005,0.118297,0.625088,0.652147,0.239409,0.725277,0.711601,0.475919,0.528706,0.102076,0.099595,0.652147
2,3,0.580290,0.217741,0.660896,0.742318,0.485083,0.485999,0.161729,0.155642,0.580290,0.668365,0.235230,0.741495,0.712418,0.465390,0.512994,0.122171,0.119364,0.668365
3,4,0.543691,0.203682,0.624297,0.759209,0.496751,0.449168,0.198410,0.188920,0.543691,0.731977,0.244811,0.805107,0.709967,0.437746,0.512751,0.126929,0.120571,0.731977
4,5,0.502140,0.188249,0.582746,0.778257,0.512182,0.411620,0.213458,0.201628,0.502140,0.718136,0.241370,0.791266,0.697712,0.468156,0.530305,0.107680,0.089192,0.718136
5,6,0.462605,0.173273,0.543211,0.792633,0.524537,0.370985,0.257053,0.251100,0.462605,0.752350,0.246185,0.825480,0.713235,0.462576,0.524055,0.100478,0.088442,0.752350
6,7,0.423869,0.159355,0.504475,0.809524,0.559932,0.339430,0.269725,0.259611,0.423869,0.839495,0.245918,0.912625,0.703431,0.437575,0.510814,0.113719,0.115009,0.839495
7,8,0.390979,0.144366,0.471586,0.818868,0.594894,0.303223,0.283472,0.282261,0.390979,0.820742,0.246327,0.893872,0.708333,0.463599,0.510006,0.097160,0.096704,0.820742
8,9,0.356357,0.133026,0.436963,0.836478,0.667846,0.283225,0.303377,0.297618,0.356357,0.897186,0.242948,0.970316,0.703431,0.461643,0.503635,0.096629,0.106106,0.897186
9,10,0.318902,0.118887,0.399508,0.847439,0.707000,0.251961,0.328500,0.321162,0.318902,0.976970,0.252647,1.050100,0.689542,0.463989,0.504364,0.081446,0.100970,0.976970


Saved: context_embedding_women_outputs/context_embedding_training_history.csv


## 11. Test Evaluation

In [31]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

display(test_metrics)

metrics_path = OUTPUT_DIR / "context_embedding_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


/tmp/ipykernel_181871/3133918950.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.689132809638977,
 'js_mean': 0.2473763973880187,
 'cross_entropy_mean': 0.7725958824157715,
 'accuracy': 0.6956521739130435,
 'macro_f1': 0.4601310300417169,
 'expected_label_mae': 0.5604319020919978,
 'entropy_pearson': 0.11507168387903947,
 'entropy_spearman': 0.10108446270995625,
 'loss': 0.6891327839303454}

Saved: context_embedding_women_outputs/context_embedding_test_metrics.json


## 12. Save Predictions

In [32]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())

predictions_path = OUTPUT_DIR / "context_embedding_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

csv_predictions_path = OUTPUT_DIR / "context_embedding_test_predictions.csv"
predictions_df.to_csv(csv_predictions_path, index=False)

print("Saved:", predictions_path)
print("Saved:", csv_predictions_path)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length,subgroup_id,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,women,63,test,men,1,Demand she pay and topple the patriarchy!,"[1.0, 0.0, 0.0]",0,0.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.3866960108280182],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and prac...","### COMMENT TO CLASSIFY\nDemand she pay and topple the patriarchy!\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societies, males hold primary power and predominate in role...",10,170,0,"[1.0, 0.0, 0.0]","[0.9352937, 0.029039739, 0.035666633]",0,0.100373,0.410067,0.066895,0.033134,0.066895
1,women,63,test,women,2,Demand she pay and topple the patriarchy!,"[1.0, 0.0, 0.0]",0,0.0,[Patriarchy],[https://en.wikipedia.org/wiki/Patriarchy],[0.4609646797180176],"[Patriarchy is a social system where men hold primary positions of authority, dominating society through various means. Sociologists attribute this to the process of socialization, which establishes gender roles and ...","### COMMENT TO CLASSIFY\nDemand she pay and topple the patriarchy!\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nPatriarchy:\nPatriarchy is a social system where men hold primary positions of authority,...",10,156,2,"[1.0, 0.0, 0.0]","[0.9250274, 0.03324514, 0.041727494]",0,0.116700,0.458491,0.077932,0.038540,0.077932
2,women,193,test,men,1,I will personally rape your mother,"[0.0, 0.0, 1.0]",2,2.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.17586159706115723],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and prac...","### COMMENT TO CLASSIFY\nI will personally rape your mother\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nMale privilege:\nIn patriarchal societies, males hold primary power and predominate in roles of le...",8,168,0,"[0.0, 0.0, 1.0]","[0.86472857, 0.05720267, 0.07806879]",0,0.213340,0.704660,2.550165,0.797930,2.550165
3,women,193,test,women,1,I will personally rape your mother,"[0.0, 0.0, 1.0]",2,2.0,[Violence against women],[https://en.wikipedia.org/wiki/Violence_against_women],[0.27236032485961914],"[Violence against women is a manifestation of historically unequal power relations between men and women, with men being the primary perpetrators due to societal patriarchal norms. This violence can take many forms, ...",### COMMENT TO CLASSIFY\nI will personally rape your mother\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against women is a manifestation of historically unequal power...,8,127,2,"[0.0, 0.0, 1.0]","[0.85829735, 0.05927819, 0.0824244]",0,0.224127,0.727644,2.495874,0.789761,2.495874
4,women,209,test,men,1,I hope she's been sterilized,"[1.0, 0.0, 0.0]",0,0.0,[Toxic masculinity],[https://en.wikipedia.org/wiki/Toxic_masculinity],[0.02771923691034317],"[Toxic masculinity refers to traditional cultural masculine norms that can be harmful to men, women, and society overall. These norms emphasize dominance, self-reliance, competition, and the restriction of emotion, o...",### COMMENT TO CLASSIFY\nI hope she's been sterilized\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nToxic masculinity:\nToxic masculinity refers to traditional cultural masculine norms that can be harmful...,9,142,0,"[1.0, 0.0, 0.0]","[0.82510126, 0.066063754, 0.10883501]",0,0.283734

Saved: context_embedding_women_outputs/context_embedding_test_predictions.parquet
Saved: context_embedding_women_outputs/context_embedding_test_predictions.csv


## 13. Standard Diagnostics

In [33]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())


Confusion matrix:
[[590   0 195]
 [ 45   0  41]
 [ 76   0 226]]

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.75      0.79       785
           1       0.00      0.00      0.00        86
           2       0.49      0.75      0.59       302

    accuracy                           0.70      1173
   macro avg       0.44      0.50      0.46      1173
weighted avg       0.68      0.70      0.68      1173


Predicted label distribution:


0    0.606138
2    0.393862
Name: proportion, dtype: float64


True label distribution:


0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

In [34]:
print("Mean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,785,0.406514,0.162583,0.142549,0.751194
1,86,2.426828,0.723364,0.197674,0.970630
2,302,0.928914,0.332238,0.040868,1.124943


Average predicted distribution by true majority label:
0 [0.7261949  0.06163635 0.21216893]
1 [0.5679621  0.07980999 0.35222772]
2 [0.39540344 0.0899471  0.51464933]


In [35]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / "context_embedding_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
2,women,581,0.682056,0.247707,0.758765,0.716007,0.482468,0.556070,0.081170,0.062271
0,men,576,0.696058,0.247194,0.788652,0.673611,0.434568,0.565320,0.145751,0.128402
1,non_binary,16,0.696811,0.241930,0.696811,0.750000,0.494949,0.542842,NaN,NaN


Saved: context_embedding_women_outputs/context_embedding_subgroup_metrics.csv


## 14. Counterfactual Subgroup Sensitivity

In [36]:
@torch.no_grad()
def predict_distribution(context_text: str, subgroup: str) -> np.ndarray:
    model.eval()

    encoding = tokenizer(
        context_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long,
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id,
    )

    return outputs["probs"].detach().cpu().numpy()[0]


### Important note on counterfactuals with context

For this context model, counterfactual subgroup sensitivity should change both:

1. the subgroup embedding, and  
2. the retrieved context.

Therefore, when comparing groups, we use the context text mapped for that `(comment_id, subgroup)` row.


In [37]:
subgroups = list(subgroup_to_id.keys())

# Lookup context text by comment_id and subgroup.
context_lookup = {
    (row["comment_id"], row["subgroup"]): row["context_input_text"]
    for _, row in context_df.iterrows()
}

unique_test_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

counterfactual_rows = []

for _, row in unique_test_comments.iterrows():
    comment_id = row["comment_id"]

    available_subgroups = [
        subgroup for subgroup in subgroups
        if (comment_id, subgroup) in context_lookup
    ]

    if len(available_subgroups) < 2:
        continue

    preds_by_group = {}

    for subgroup in available_subgroups:
        context_text = context_lookup[(comment_id, subgroup)]
        preds_by_group[subgroup] = predict_distribution(context_text, subgroup)

    pairwise_js = []

    for group_a, group_b in itertools.combinations(available_subgroups, 2):
        js = jensenshannon(
            preds_by_group[group_a],
            preds_by_group[group_b],
            base=2,
        ) ** 2

        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": comment_id,
        "text": row["text"],
        "n_available_subgroups": len(available_subgroups),
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Counterfactual rows:", cf_df.shape)
print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(
    cf_df
    .sort_values("max_pairwise_js", ascending=False)
    .head(30)
)

cf_path = OUTPUT_DIR / "context_embedding_counterfactual_js.csv"
cf_df.to_csv(cf_path, index=False)
print("Saved:", cf_path)


Counterfactual rows: (582, 5)
Mean counterfactual JS: 0.0025119022668322836
Median counterfactual JS: 0.0015572206071770654
Mean max-pairwise JS: 0.0025289596096266416
Median max-pairwise JS: 0.0015786025517877528


,comment_id,text,n_available_subgroups,mean_pairwise_js,max_pairwise_js
536,44042,She's so retarded,2,0.055990,0.055990
299,22181,I will suck on yo clitoris and finger you at the same time eat yo pussy from the back let you ride my face till you cum and go home and post my bitch,2,0.030867,0.030867
559,47822,Survey says women are emotional lunatics.,2,0.028663,0.028663
413,32441,OOMFIES TRYNNA SEND ME TO JAIL SHE AINT SHIT,2,0.028116,0.028116
214,14803,"The ultimate cum target, now with all that cream dripping off her...",2,0.027224,0.027224
576,49722,THESE 4 WOMEN WHO HATE THIS COUNTRY USING THE RACIST CARD EVERYDAY THESE 4 WOMEN HATE AMERICA THEY ARE SOCIALISTS RACIST DISGUSTING DISGRACE TO AMERICA DEMOCRATIC HAVE LOST IT ....TRUMP IS CORRECT OMAR TALKS HORRIBLE...,2,0.026834,0.026834
506,40084,All four of these skanky narcissistic feminists need feeding to lions.,2,0.025426,0.025426
21,1572,I guess a Pussy Hat isn't the stupidest thing I've ever seen on a woman's head.,2,0.024703,0.024703
83,5410,Them - DON'T YOU DARE OBJECTIFY US!! ​ Also them - Gets publicly naked as much as fucking possible.,2,0.024358,0.024358
86,5650,Was she a former prostitute that Richard Gere was making into a lady?,2,0.021889,0.021889


Saved: context_embedding_women_outputs/context_embedding_counterfactual_js.csv


## 15. Men vs Women Counterfactual Analysis


In [38]:
def pairwise_counterfactual_js(group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    unique_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

    for _, row in unique_comments.iterrows():
        comment_id = row["comment_id"]

        if (comment_id, group_a) not in context_lookup:
            continue

        if (comment_id, group_b) not in context_lookup:
            continue

        context_a = context_lookup[(comment_id, group_a)]
        context_b = context_lookup[(comment_id, group_b)]

        pred_a = predict_distribution(context_a, group_a)
        pred_b = predict_distribution(context_b, group_b)

        js = jensenshannon(
            pred_a,
            pred_b,
            base=2,
        ) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row["text"],
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
            f"context_{group_a}": context_a,
            f"context_{group_b}": context_b,
        })

    return pd.DataFrame(rows)


print("Available subgroups:")
print(subgroup_to_id)


Available subgroups:
{'men': 0, 'non_binary': 1, 'women': 2}


In [39]:
if "men" in subgroup_to_id and "women" in subgroup_to_id:
    men_women_df = pairwise_counterfactual_js(
        "men",
        "women",
    )

    print("Men vs women rows:", men_women_df.shape)
    print("Men vs women mean JS:", men_women_df["js"].mean())
    print("Men vs women median JS:", men_women_df["js"].median())

    display(
        men_women_df
        .sort_values("js", ascending=False)
        .head(30)
    )

    men_women_path = OUTPUT_DIR / "context_embedding_men_vs_women_counterfactual.csv"
    men_women_df.to_csv(men_women_path, index=False)
    print("Saved:", men_women_path)


Men vs women rows: (575, 9)
Men vs women mean JS: 0.0024736695908826514
Men vs women median JS: 0.001573689625515254


,comment_id,text,group_a,group_b,js,pred_men,pred_women,context_men,context_women
531,44042,She's so retarded,men,women,0.055990,"[0.86920017, 0.048626017, 0.08217389]","[0.63722557, 0.09344671, 0.26932773]","### COMMENT TO CLASSIFY\nShe's so retarded\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nGender role:\nGender roles are social norms that vary across cultures, influencing a wide range of human behavior, ...","### COMMENT TO CLASSIFY\nShe's so retarded\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nSexism:\nSexism is prejudice or discrimination based on one's sex or gender, primarily affecting women and girls...."
295,22181,I will suck on yo clitoris and finger you at the same time eat yo pussy from the back let you ride my face till you cum and go home and post my bitch,men,women,0.030867,"[0.66312146, 0.13699687, 0.19988167]","[0.47413582, 0.15281728, 0.3730469]",### COMMENT TO CLASSIFY\nI will suck on yo clitoris and finger you at the same time eat yo pussy from the back let you ride my face till you cum and go home and post my bitch\n\n### ANNOTATOR GENDER\nmen\n\n### RETRI...,### COMMENT TO CLASSIFY\nI will suck on yo clitoris and finger you at the same time eat yo pussy from the back let you ride my face till you cum and go home and post my bitch\n\n### ANNOTATOR GENDER\nwomen\n\n### RET...
553,47822,Survey says women are emotional lunatics.,men,women,0.028663,"[0.6869857, 0.08711885, 0.2258955]","[0.49798942, 0.105275035, 0.39673558]","### COMMENT TO CLASSIFY\nSurvey says women are emotional lunatics.\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nGender role:\nGender roles are social norms that vary across cultures, influencing a wide r...",### COMMENT TO CLASSIFY\nSurvey says women are emotional lunatics.\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nMisogyny:\nMisogyny is a form of sexism that perpetuates women's subordinate status in pa...
213,14803,"The ultimate cum target, now with all that cream dripping off her...",men,women,0.027224,"[0.60616344, 0.1330339, 0.26080266]","[0.4239474, 0.14434677, 0.43170586]","### COMMENT TO CLASSIFY\nThe ultimate cum target, now with all that cream dripping off her...\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nHegemonic masculinity:\nHegemonic masculinity refers to the cult...","### COMMENT TO CLASSIFY\nThe ultimate cum target, now with all that cream dripping off her...\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against women is a manifesta..."
569,49722,THESE 4 WOMEN WHO HATE THIS COUNTRY USING THE RACIST CARD EVERYDAY THESE 4 WOMEN HATE AMERICA THEY ARE SOCIALISTS RACIST DISGUSTING DISGRACE TO AMERICA DEMOCRATIC HAVE LOST IT ....TRUMP IS CORRECT OMAR TALKS HORRIBLE...,men,women,0.026834,"[0.67713606, 0.08931171, 0.23355219]","[0.4926351, 0.109991975, 0.39737293]",### COMMENT TO CLASSIFY\nTHESE 4 WOMEN WHO HATE THIS COUNTRY USING THE RACIST CARD EVERYDAY THESE 4 WOMEN HATE AMERICA THEY ARE SOCIALISTS RACIST DISGUSTING DISGRACE TO AMERICA DEMOCRATIC HAVE LOST IT ....TRUMP IS CO...,### COMMENT TO CLASSIFY\nTHESE 4 WOMEN WHO HATE THIS COUNTRY USING THE RACIST CARD EVERYDAY THESE 4 WOMEN HATE AMERICA THEY ARE SOCIALISTS RACIST DISGUSTING DISGRACE TO AMERICA DEMOCRATIC HAVE LOST IT ....TRUMP IS CO...
501,40084,All four of these skanky narcissistic feminists need feeding to lions.,men,women,0.025426,"[0.67182314, 0.09951928, 0.22865759]","[0.49535072, 0.114421815, 0.39022747]",### COMMENT TO CLASSIFY\nAll four of these skanky narcissistic feminists need feeding to lions.\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BACKGROUND\nHegemonic masculinity:\nHegemonic masculinity refers to the cu...,### COMMENT TO CLASSIFY\nAll four of these skanky narcissistic feminists need feeding to lions.\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED BACKGROUND\nViolence against women:\nViolence against women is a manifes...
21,1572,I guess a Pussy Hat isn't the stupidest thing I'

Saved: context_embedding_women_outputs/context_embedding_men_vs_women_counterfactual.csv


## 16. True Men/Women Disagreement vs Model Counterfactual Disagreement


In [40]:
def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(parse_distribution(x), dtype=float)
    except Exception:
        return False
    if arr.shape[0] != NUM_LABELS:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


def true_pairwise_disagreement_from_context_df(
    full_df: pd.DataFrame,
    group_a: str,
    group_b: str,
) -> pd.DataFrame:
    rows = []

    grouped = full_df.groupby("comment_id")

    for comment_id, group in grouped:
        if group_a not in set(group["subgroup"]):
            continue
        if group_b not in set(group["subgroup"]):
            continue

        row_a = group[group["subgroup"] == group_a].iloc[0]
        row_b = group[group["subgroup"] == group_b].iloc[0]

        dist_a = parse_distribution(row_a["target_distribution"])
        dist_b = parse_distribution(row_b["target_distribution"])

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)

        true_js = jensenshannon(dist_a, dist_b, base=2) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row_a["text"],
            "true_js": float(true_js),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


In [41]:
if "men" in subgroup_to_id and "women" in subgroup_to_id:
    true_df = true_pairwise_disagreement_from_context_df(
        context_df,
        "men",
        "women",
    )

    comparison_df = true_df.merge(
        men_women_df[[
            "comment_id",
            "js",
            "pred_men",
            "pred_women",
        ]],
        on="comment_id",
        how="inner",
    )

    comparison_df = comparison_df.rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true men/women overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())

    if len(nonzero) > 0:
        print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
        print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["men_true_label"].astype(str)
        + "-"
        + comparison_df["women_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(
            n=("comment_id", "count"),
            mean_true_js=("true_js", "mean"),
            mean_model_js=("model_js", "mean"),
        )
        .sort_values("mean_true_js", ascending=False)
    )

    display(
        comparison_df
        .sort_values("true_js", ascending=False)
        .head(30)
    )

    comparison_path = OUTPUT_DIR / "context_embedding_true_vs_model_men_women.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("Saved:", comparison_path)


N true men/women overlap: 3869
N comparison: 575
Mean true JS: 0.34650682259322935
Mean model JS: 0.0024736695908826514
Mean capture ratio, true_js > 0: 0.009357285623941428
Median capture ratio, true_js > 0: 0.0020045189302639586


,n,mean_true_js,mean_model_js
label_pair,,,
1-0,21,1.000000,0.001736
2-1,7,1.000000,0.003521
0-1,32,0.962852,0.002893
2-0,50,0.865213,0.003160
0-2,84,0.828284,0.003321
1-2,12,0.827820,0.002298
0-0,291,0.055885,0.002073
1-1,6,0.051880,0.002758
2-2,72,0.015082,0.002560


,comment_id,text,true_js,men_true_dist,women_true_dist,men_true_label,women_true_label,model_js,pred_men,pred_women,label_pair
480,38410,to all those ugly hoes complaining about the recent unedited photoshoot of nct. Fuck off you should look at yourself. They are also fucking humans with feelings and skin problems. for example I also don't have clear ...,1.0,"[0.0, 1.0, 0.0]","[1.0, 0.0, 0.0]",1,0,0.002741,"[0.2966474, 0.09915655, 0.6041961]","[0.2443811, 0.095514864, 0.660104]",1-0
183,12324,Now she most certainly is on tape saying she wanted to grab women by the pussy right? No?,1.0,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.002469,"[0.95446795, 0.02121459, 0.024317538]","[0.9269705, 0.03338967, 0.039639972]",0-1
478,38126,"Y'all always slip up and say how y'all see women period because this includes the majority of women in yall lives that's not related to you. Especially since men demand pussy just from taking you out, selling you wee...",1.0,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.001867,"[0.5652295, 0.10669716, 0.32807332]","[0.51573104, 0.11212705, 0.37214193]",0-1
192,13332,"I'd fuck you like a bitch, pulling your hair and spanking your ass as I stretch your tight little hole",1.0,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.002988,"[0.33905867, 0.120326504, 0.54061484]","[0.28269634, 0.11701019, 0.60029346]",0-1
476,37948,"Ya the prime example of an unloyal Girlfriend you, and a fuckin pussy",1.0,"[0.6666666865348816, 0.0, 0.3333333432674408]","[0.0, 1.0, 0.0]",0,1,0.003936,"[0.5968199, 0.15012251, 0.25305754]","[0.52779466, 0.15888472, 0.31332064]",0-1
194,13371,women enjoy retarded expressions of dumb niceness,1.0,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]",0,2,0.003890,"[0.93013287, 0.030231528, 0.03963553]","[0.8889686, 0.042753596, 0.06827788]",0-2
196,13431,I'd spin you around and slam you on this dick like a real fuck toy.,1.0,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]",2,0,0.000815,"[0.66871953, 0.13976434, 0.19151616]","[0.63793993, 0.14721887, 0.21484117]",2-0
474,37899,"U hoes go out 2 Eat with anybody, you gone get kidnapped 1 day u hungry bitch",1.0,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]",2,0,0.002257,"[0.28365228, 0.09788746, 0.61846024]","[0.23724608, 0.093395084, 0.6693588]",2-0
201,13955,Depression Stress wait no women have those too yea this guys a dickbag,1.0,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]",2,0,0.000099,"[0.8229677, 0.07123673, 0.10579562]","[0.8168442, 0.070074044, 0.11308165]",2-0
203,14011,Plant her face in my wife's pussy while I fuck her from behind,1.0,"[0.0, 1.0, 0.0]","[1.0, 0.0, 0.0]",1,0,0.000351,"[0.9081269, 0.044973638, 0.04689947]","[0.8951849, 0.050049108, 0.054766018]",1-0


Saved: context_embedding_women_outputs/context_embedding_true_vs_model_men_women.csv


## 17. Inspect High-Impact Context Examples

In [42]:
if "men_women_df" in globals() and len(men_women_df) > 0:
    top_row = men_women_df.sort_values("js", ascending=False).iloc[0]

    print("Comment:")
    print(top_row["text"])

    print("\nPrediction men:")
    print(top_row["pred_men"])

    print("\nPrediction women:")
    print(top_row["pred_women"])

    print("\nContext men:")
    print(top_row["context_men"])

    print("\nContext women:")
    print(top_row["context_women"])


Comment:
She's so retarded

Prediction men:
[0.86920017 0.04862602 0.08217389]

Prediction women:
[0.63722557 0.09344671 0.26932773]

Context men:
### COMMENT TO CLASSIFY
She's so retarded

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Gender role:
Gender roles are social norms that vary across cultures, influencing a wide range of human behavior, including profession, relationships, and personal expression. Traditionally, women have been confined to the "private" sphere while men occupy the "public" sphere. Sociologists distinguish between gender role and sex identity, with the former being shaped by societal expectations and the latter being an individual's internal sense of their own gender. Theories such as social constructionism argue that differences in behavior are due to cultural and social factors rather than biological or essentialist ones.

Context women:
### COMMENT TO CLASSIFY
She's so retarded

### ANNOTATOR GENDER
women

### RETRIEVED BACKGROUND
Sexism:
Sexism is pr

## 18. Interpretation Guide

Compare this model against:

1. no-context subgroup embedding baseline,
2. strong FiLM no-context baseline,
3. later context + token model,
4. later context + strong FiLM model.

Important comparisons:

```text
1. Overall KL / JS
2. Macro F1
3. Class-2 recall
4. Whether class 1 is ever predicted
5. Mean counterfactual JS
6. Men↔women model JS
7. True-vs-model men/women capture ratio
```

A useful result is not only lower KL. If context increases counterfactual subgroup sensitivity without destroying overall metrics, that still supports the dissertation hypothesis.


In [43]:
context_df["context_input_text"].nunique()

7958

In [44]:
len(context_df)

7962

In [45]:
context_df["text"].nunique()

3951

In [46]:
for i in range(5):
    print("="*80)
    print(context_df.iloc[i]["context_input_text"])

### COMMENT TO CLASSIFY
First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)

### ANNOTATOR GENDER
men

### RETRIEVED BACKGROUND
Male privilege:
In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and practical advantages over women. These privileges are often invisible to holders, leading them to attribute their status to individual merits rather than unearned advantages. The ideal masculine norm varies by society but is often associated with being white, heterosexual, stoic, wealthy, strong, tough, competitive, and autonomous. Men who deviate from this norm may not benefit fully from privilege, while those who conform to it are more likely to receive rewards and recognition.
### COMMENT TO CLASSIFY
First off you look cool as fuck! Anyway if we 